# Import libraries

In [41]:
from pathlib import Path
import pandas as pd
import json
import re

# Logging Setup

In [42]:
# Current working directory
BASE_PATH = Path.cwd().parent

DATA_PATH = BASE_PATH / "data" / "raw"/ "app_logs_7days.jsonl"
QUARANTINE_PATH = BASE_PATH / "notebooks" / "outputs" / "malformed_records.jsonl"

# Data Loading

Here I used **Lazy file streaming**. Why I think that's matter: 
- Python processes this log file sequentially line-by-line rather than loading an entire file into RAM
- This ensures memory usage stays contanst O(1), no matter the size of our `.jsonl` file 

In [43]:
success_count = 0
error_count = 0
valid_records = []

with open(DATA_PATH, mode="r", encoding="utf-8") as f_in, \
	open(QUARANTINE_PATH, mode="w", encoding="utf-8") as f_err: # overwrite mode
		# Stream line-by-line
		for line_num, line in enumerate(f_in, start=1):
			raw_line = line.strip() # <class 'str'>
		   
			if not raw_line:
				continue 
			
			try:
				record = json.loads(raw_line) # <class 'dict'>
				success_count += 1
				valid_records.append(record)
			
			except json.JSONDecodeError as e:
				error_count += 1
				
				malformed_record = {
					"source_file": str(DATA_PATH),
					"line_number": line_num,
					"raw_line": raw_line,
					"error_message": str(e)
				}
				f_err.write(json.dumps(malformed_record) + "\n")

# EDA

In [44]:
df = pd.DataFrame(valid_records)
del valid_records 

In [45]:
df.head()

,timestamp,service,level,message,request_id,trace_id
0,2026-07-27T00:02:47Z,notification-worker,ERROR,ERR SMTPConnRefused host=mail-gw,req-65568711,NaN
1,2026-07-27T00:12:06Z,auth-service,INFO,Session created uid=u2746,req-72350830,NaN
2,2026-07-27T00:13:20Z,payment-api,WARN,Retry 1/3 calling notification-worker,req-22315507,NaN
3,2026-07-27T00:20:17Z,notification-worker,INFO,SMS sent uid=u1132,req-27740248,NaN
4,2026-07-27T00:27:38Z,auth-service,WARN,Slow login 900ms uid=u7882,req-95306788,NaN


In [46]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2905 entries, 0 to 2904
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   timestamp   2905 non-null   str  
 1   service     2905 non-null   str  
 2   level       2887 non-null   str  
 3   message     2905 non-null   str  
 4   request_id  2905 non-null   str  
 5   trace_id    1199 non-null   str  
dtypes: str(6)
memory usage: 136.3 KB


In [47]:
non_null_counts = df.notnull().sum()
null_counts = df.isnull().sum()
null_percentage = (null_counts / len(df)) * 100

summary_df = pd.DataFrame({
	"Non-null Count": non_null_counts,
	"Null Count": null_counts,
	"Null Percentage (%)": null_percentage.round(2)
})

print(summary_df)

            Non-null Count  Null Count  Null Percentage (%)
timestamp             2905           0                 0.00
service               2905           0                 0.00
level                 2887          18                 0.62
message               2905           0                 0.00
request_id            2905           0                 0.00
trace_id              1199        1706                58.73


**Exact Row Duplicates** occured

In [48]:
duplicated_df = df[df.duplicated(keep='first')]

display(duplicated_df)
print(len(duplicated_df), "duplicated records found.")

,timestamp,service,level,message,request_id,trace_id
57,2026-07-27T03:51:26Z,batch-report,WARN,Report row mismatch expected=1061 got=1082,req-84892608,NaN
90,2026-07-27T05:52:54Z,auth-service,INFO,Token refreshed uid=u9837,req-25014631,NaN
209,2026-07-27T12:48:15Z,notification-worker,INFO,Email sent uid=u3328,req-33562830,NaN
321,2026-07-27T19:44:26Z,auth-service,INFO,Session created uid=u1998,req-77491435,NaN
372,2026-07-27T22:23:19Z,batch-report,INFO,Daily report job finished rows=851,req-69920305,NaN
474,2026-07-28T04:54:18Z,payment-api,INFO,Payment processed txn=t141094 amount=120000,req-79342339,NaN
535,2026-07-28T10:14:59Z,auth-service,INFO,Token refreshed uid=u5872,req-75893962,NaN
766,2026-07-28T23:38:39Z,batch-report,INFO,Daily report job finished rows=1163,req-75050252,NaN
824,2026-07-29T01:39:57Z,batch-report,INFO,Daily report job finished rows=1110,req-92220752,NaN
842,2026-07-29T02:44:06Z,notification-worker,INFO,Email sent uid=u7013,req-66736822,NaN


28 duplicated records found.


In [49]:
df_cleaned = df.drop_duplicates(keep='first')

## 1. timestamp

In [50]:
def detect_ts_format(x):
    x = str(x)

    if re.fullmatch(r"\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}Z", x):
        return "UTC_Z"                     # 2026-07-27T08:30:00Z

    elif re.fullmatch(r"\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}[+-]\d{2}:\d{2}", x):
        return "ISO_timezone_offset"        # 2026-07-27T07:58:15+07:00

    elif re.fullmatch(r"\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}", x):
        return "datetime_space"             # 2026-07-27 08:30:00

    elif re.fullmatch(r"\d{4}-\d{2}-\d{2}", x):
        return "date_only"                  # 2026-07-27

    else:
        return "unknown/invalid"

df_cleaned["timestamp_format"] = df_cleaned["timestamp"].apply(detect_ts_format)

print(df_cleaned["timestamp_format"].value_counts())

timestamp_format
UTC_Z                  2267
ISO_timezone_offset     590
unknown/invalid          20
Name: count, dtype: int64


In [51]:
df_cleaned[df_cleaned['timestamp_format'] == 'unknown/invalid']

,timestamp,service,level,message,request_id,trace_id,timestamp_format
33,not-a-date,auth-service,WARN,Clock sync failed,req-32170750,NaN,unknown/invalid
228,not-a-date,auth-service,WARN,Clock sync failed,req-80542438,trace-2840524582,unknown/invalid
483,not-a-date,auth-service,WARN,Clock sync failed,req-64770806,trace-3597287435,unknown/invalid
502,not-a-date,batch-report,WARN,Clock sync failed,req-89141563,trace-1842573576,unknown/invalid
553,not-a-date,notification-worker,WARN,Clock sync failed,req-95756894,trace-1839947796,unknown/invalid
696,not-a-date,payment-api,WARN,Clock sync failed,req-46941691,NaN,unknown/invalid
1029,not-a-date,batch-report,WARN,Clock sync failed,req-58571652,trace-8069263805,unknown/invalid
1191,not-a-date,batch-report,WARN,Clock sync failed,req-82179030,trace-5297869651,unknown/invalid
1243,not-a-date,web-portal,WARN,Clock sync failed,req-14251180,trace-2529141801,unknown/invalid
1349,not-a-date,web-portal,WARN,Clock sync failed,req-62013608,NaN,unknown/invalid


In [52]:
df_cleaned["timezone_offset"] = df_cleaned["timestamp"].str.extract(
    r"([+-]\d{2}:\d{2})$"
)

print(df_cleaned["timezone_offset"].value_counts())

timezone_offset
+07:00    590
Name: count, dtype: int64


In [53]:
display(df_cleaned[df_cleaned["timezone_offset"].notna()])

,timestamp,service,level,message,request_id,trace_id,timestamp_format,timezone_offset
119,2026-07-27T07:58:15+07:00,web-portal,INFO,Request completed path=/report in 554ms,req-20952896,NaN,ISO_timezone_offset,+07:00
125,2026-07-27T08:23:40+07:00,web-portal,INFO,Request completed path=/home in 752ms,req-70210270,NaN,ISO_timezone_offset,+07:00
126,2026-07-27T08:25:30+07:00,web-portal,INFO,Request completed path=/home in 489ms,req-31306555,NaN,ISO_timezone_offset,+07:00
133,2026-07-27T08:53:59+07:00,web-portal,INFO,Request completed path=/report in 768ms,req-48260510,NaN,ISO_timezone_offset,+07:00
137,2026-07-27T09:10:52+07:00,web-portal,INFO,Request completed path=/home in 535ms,req-91424562,NaN,ISO_timezone_offset,+07:00
...,...,...,...,...,...,...,...,...
2900,2026-08-03T06:15:30+07:00,web-portal,WARN,Response time 2100ms path=/report,req-45353565,trace-8774772376,ISO_timezone_offset,+07:00
2901,2026-08-03T06:30:10+07:00,web-portal,ERROR,ERR HTTP 502 upstream=payment-api path=/checkout,req-99609973,trace-7354458743,ISO_timezone_offset,+07:00
2902,2026-08-03T06:43:40+07:00,web-portal,INFO,Request completed path=/report in 284ms,req-87206285,trace-5698353763,ISO_timezone_offset,+07:00
2903,2026-08-03T06:55:40+07:00,web-portal,INFO,Request completed path=/home in 871ms,req-72913088,trace-5937414387,ISO_timezone_offset,+07:00


Time range confirmation: 7 days from 27/07/2026 to 02/08/2026

In [54]:
df_cleaned['event_date_utc'] = pd.to_datetime(df_cleaned['timestamp'], utc=True, errors='coerce').dt.date
display(df_cleaned['event_date_utc'])

print(type(df_cleaned['event_date_utc'].iloc[0])) # first row

0       2026-07-27
1       2026-07-27
2       2026-07-27
3       2026-07-27
4       2026-07-27
           ...    
2900    2026-08-02
2901    2026-08-02
2902    2026-08-02
2903    2026-08-02
2904    2026-08-02
Name: event_date_utc, Length: 2877, dtype: object

<class 'datetime.date'>


--- 
Let's go a bit deeper! (or I can say I'm doing: Root Cause Analysis)

In [55]:
not_a_date_df = df_cleaned[df_cleaned['timestamp'] == 'not-a-date'].copy()
not_a_date_df

,timestamp,service,level,message,request_id,trace_id,timestamp_format,timezone_offset,event_date_utc
33,not-a-date,auth-service,WARN,Clock sync failed,req-32170750,NaN,unknown/invalid,NaN,NaT
228,not-a-date,auth-service,WARN,Clock sync failed,req-80542438,trace-2840524582,unknown/invalid,NaN,NaT
483,not-a-date,auth-service,WARN,Clock sync failed,req-64770806,trace-3597287435,unknown/invalid,NaN,NaT
502,not-a-date,batch-report,WARN,Clock sync failed,req-89141563,trace-1842573576,unknown/invalid,NaN,NaT
553,not-a-date,notification-worker,WARN,Clock sync failed,req-95756894,trace-1839947796,unknown/invalid,NaN,NaT
696,not-a-date,payment-api,WARN,Clock sync failed,req-46941691,NaN,unknown/invalid,NaN,NaT
1029,not-a-date,batch-report,WARN,Clock sync failed,req-58571652,trace-8069263805,unknown/invalid,NaN,NaT
1191,not-a-date,batch-report,WARN,Clock sync failed,req-82179030,trace-5297869651,unknown/invalid,NaN,NaT
1243,not-a-date,web-portal,WARN,Clock sync failed,req-14251180,trace-2529141801,unknown/invalid,NaN,NaT
1349,not-a-date,web-portal,WARN,Clock sync failed,req-62013608,NaN,unknown/invalid,NaN,NaT


- This reveals a critical pattern: **every single invalid record has the exact same message: "Clock sync failed"**
- The issue was spread across all 5 services (`auth-service`, `batch-report`, `notification-worker`, `payment-api`, and `web-portal`), meaning this is likely an underlying infrastructure or host-level issue rather than an isolated application bug

Applying **Data Imputation** technique: For `ffill` to be valid, the time gap between the record before the failure and the record after the failure must be narrow (e.g., milliseconds to a few seconds). If the gap is small, `ffill` introduces negligible error.

In [56]:
target_positions = [
    position
    for position, value in enumerate(df_cleaned["timestamp"])
    if value == "not-a-date"
]

context_positions = sorted({
    nearby_position
    for position in target_positions
    for nearby_position in range(
        max(0, position - 2),
        min(len(df_cleaned), position + 3),
    )
})

result = df_cleaned.iloc[context_positions].copy()
result["is_not_a_date"] = result["timestamp"].eq("not-a-date")

pd.set_option("display.max_rows", 100)
result

,timestamp,service,level,message,request_id,trace_id,timestamp_format,timezone_offset,event_date_utc,is_not_a_date
31,2026-07-27T02:43:27Z,payment-api,INFO,Payment processed txn=t919488 amount=250000,req-97219599,NaN,UTC_Z,NaN,2026-07-27,False
32,2026-07-27T02:44:22Z,auth-service,INFO,User login success uid=u3442,req-29876811,NaN,UTC_Z,NaN,2026-07-27,False
33,not-a-date,auth-service,WARN,Clock sync failed,req-32170750,NaN,unknown/invalid,NaN,NaT,True
34,2026-07-27T02:44:29Z,payment-api,INFO,Balance check ok uid=u3306,req-84801233,NaN,UTC_Z,NaN,2026-07-27,False
35,2026-07-27T02:49:58Z,notification-worker,WARN,Queue depth high depth=863,req-99929175,NaN,UTC_Z,NaN,2026-07-27,False
226,2026-07-27T13:54:41+07:00,web-portal,INFO,Request completed path=/report in 550ms,req-22086184,NaN,ISO_timezone_offset,+07:00,2026-07-27,False
227,2026-07-27T13:59:26+07:00,web-portal,INFO,Request completed path=/report in 119ms,req-78140959,NaN,ISO_timezone_offset,+07:00,2026-07-27,False
228,not-a-date,auth-service,WARN,Clock sync failed,req-80542438,trace-2840524582,unknown/invalid,NaN,NaT,True
229,2026-07-27T14:02:58Z,notification-worker,INFO,SMS sent uid=u2847,req-11619606,NaN,UTC_Z,NaN,2026-07-27,False
230,2026-07-27T14:08:30Z,batch-report,INFO,Daily report job started,req-59805178,NaN,UTC_Z,NaN,2026-07-27,False


In [70]:
expected_corrupted = (df['timestamp'] == "not-a-date")
expected_corrupted

0       False
1       False
2       False
3       False
4       False
        ...  
2900    False
2901    False
2902    False
2903    False
2904    False
Name: timestamp, Length: 2905, dtype: bool

## 2. service

In [57]:
df_cleaned['service'].unique()

<StringArray>
['notification-worker',        'auth-service',         'payment-api',
        'batch-report',          'web-portal']
Length: 5, dtype: str

## 3. level

In [58]:
df_cleaned['level'].unique()

<StringArray>
['ERROR', 'INFO', 'WARN', nan]
Length: 4, dtype: str

18 missing values out of 2905 records

In [59]:
df_cleaned[df_cleaned['level'].isna()]

,timestamp,service,level,message,request_id,trace_id,timestamp_format,timezone_offset,event_date_utc
10,2026-07-30T12:07:36Z,notification-worker,NaN,Heartbeat ok,req-48936328,NaN,UTC_Z,NaN,2026-07-30
94,2026-07-30T18:23:58Z,payment-api,NaN,Heartbeat ok,req-30906603,NaN,UTC_Z,NaN,2026-07-30
213,2026-07-30T13:53:09Z,payment-api,NaN,Heartbeat ok,req-89098923,NaN,UTC_Z,NaN,2026-07-30
300,2026-07-31T18:38:08Z,auth-service,NaN,Heartbeat ok,req-18963036,trace-4082960786,UTC_Z,NaN,2026-07-31
697,2026-08-01T01:47:50Z,payment-api,NaN,Heartbeat ok,req-21881015,trace-7088048157,UTC_Z,NaN,2026-08-01
703,2026-08-01T07:53:41+07:00,web-portal,NaN,Heartbeat ok,req-71725422,trace-9264884678,ISO_timezone_offset,+07:00,2026-08-01
830,2026-08-02T01:19:22Z,auth-service,NaN,Heartbeat ok,req-11122998,trace-7153186561,UTC_Z,NaN,2026-08-02
1038,2026-07-29T03:14:59Z,batch-report,NaN,Heartbeat ok,req-22252693,NaN,UTC_Z,NaN,2026-07-29
1277,2026-07-28T18:01:01Z,auth-service,NaN,Heartbeat ok,req-29057175,NaN,UTC_Z,NaN,2026-07-28
1316,2026-07-27T23:43:39Z,batch-report,NaN,Heartbeat ok,req-41470269,NaN,UTC_Z,NaN,2026-07-27


## 4. message

In [60]:
df_cleaned['message'].unique()

<StringArray>
[        'ERR SMTPConnRefused host=mail-gw',
                'Session created uid=u2746',
    'Retry 1/3 calling notification-worker',
                       'SMS sent uid=u1132',
               'Slow login 900ms uid=u7882',
                'Session created uid=u2528',
                     'Email sent uid=u2106',
                 'Daily report job started',
 'Report row mismatch expected=843 got=759',
                       'SMS sent uid=u5942',
 ...
  'Request completed path=/report in 167ms',
  'Request completed path=/report in 643ms',
    'Request completed path=/home in 753ms',
    'Request completed path=/home in 178ms',
  'Request completed path=/report in 332ms',
    'Request completed path=/home in 795ms',
    'Request completed path=/home in 868ms',
  'Request completed path=/report in 284ms',
    'Request completed path=/home in 871ms',
  'Request completed path=/report in 109ms']
Length: 2153, dtype: str

High cardinality: 2,153 distinct strings / 2,905 rows, because messages embed dynamic values such as user IDs, transaction IDs, amounts, paths, latency, row counts, etc

---

Deep dive in ERROR rows

In [61]:
df_cleaned[df_cleaned['level'] == 'ERROR']['message'].unique()

<StringArray>
[                'ERR SMTPConnRefused host=mail-gw',
     'ERR ConnTimeout db-primary after 30s retry=3',
          'ERR PaymentDeclined txn=t811163 code=51',
  'ERR NullPointer in ReportBuilder step=aggregate',
                   'ERR AuthTokenExpired uid=u6576',
          'ERR PaymentDeclined txn=t108105 code=51',
 'ERR HTTP 502 upstream=payment-api path=/checkout',
          'ERR PaymentDeclined txn=t534303 code=51',
                   'ERR AuthTokenExpired uid=u7209',
          'ERR PaymentDeclined txn=t108060 code=51',
                   'ERR AuthTokenExpired uid=u6592',
          'ERR PaymentDeclined txn=t922041 code=51',
                   'ERR AuthTokenExpired uid=u8751',
          'ERR PaymentDeclined txn=t642892 code=51',
                   'ERR AuthTokenExpired uid=u1587',
                   'ERR AuthTokenExpired uid=u8324',
                   'ERR AuthTokenExpired uid=u4389',
                   'ERR AuthTokenExpired uid=u5550',
                   'ERR AuthToke

Only ERROR logs have messages starting with the keyword 'ERR'

In [66]:
df_cleaned[
    df_cleaned['level'] != 'ERROR']

,timestamp,service,level,message,request_id,trace_id,timestamp_format,timezone_offset,event_date_utc,error_type
1,2026-07-27T00:12:06Z,auth-service,INFO,Session created uid=u2746,req-72350830,NaN,UTC_Z,NaN,2026-07-27,NaN
2,2026-07-27T00:13:20Z,payment-api,WARN,Retry 1/3 calling notification-worker,req-22315507,NaN,UTC_Z,NaN,2026-07-27,NaN
3,2026-07-27T00:20:17Z,notification-worker,INFO,SMS sent uid=u1132,req-27740248,NaN,UTC_Z,NaN,2026-07-27,NaN
4,2026-07-27T00:27:38Z,auth-service,WARN,Slow login 900ms uid=u7882,req-95306788,NaN,UTC_Z,NaN,2026-07-27,NaN
5,2026-07-27T00:32:14Z,auth-service,INFO,Session created uid=u2528,req-38688676,NaN,UTC_Z,NaN,2026-07-27,NaN
...,...,...,...,...,...,...,...,...,...,...
2899,2026-08-03T06:06:45+07:00,web-portal,INFO,Request completed path=/home in 868ms,req-17243541,trace-5560561097,ISO_timezone_offset,+07:00,2026-08-02,NaN
2900,2026-08-03T06:15:30+07:00,web-portal,WARN,Response time 2100ms path=/report,req-45353565,trace-8774772376,ISO_timezone_offset,+07:00,2026-08-02,NaN
2902,2026-08-03T06:43:40+07:00,web-portal,INFO,Request completed path=/report in 284ms,req-87206285,trace-5698353763,ISO_timezone_offset,+07:00,2026-08-02,NaN
2903,2026-08-03T06:55:40+07:00,web-portal,INFO,Request completed path=/home in 871ms,req-72913088,trace-5937414387,ISO_timezone_offset,+07:00,2026-08-02,NaN


## 5. request_id

In [ ]:
df_cleaned['request_id'].head(10)

Check to see all match `req-########`

In [ ]:
pattern = r"req-\d{8}"

# Rows that do not match the pattern
df_cleaned[~df_cleaned['request_id'].str.fullmatch(pattern, na=False)]

In [ ]:
duplicated_request_df = df_cleaned[df_cleaned.duplicated(subset=['request_id'], keep='first')]
display(duplicated_request_df)

print(len(duplicated_request_df), "duplicated request_id records found.")

## 6. trace_id

In [ ]:
len(df_cleaned)

In [ ]:
df_cleaned[~df_cleaned['trace_id'].isna()]

In [ ]:
pattern = r"trace-\d{10}"

# Rows that do not match the pattern (exclude NaN values)
df_cleaned[~df_cleaned['trace_id'].str.fullmatch(pattern, na=True)]

Let's check when the `trace_id` starts existing

In [ ]:
missing_trace_df = df_cleaned[df_cleaned['trace_id'].isna()]
missing_trace_df[missing_trace_df['event_date_utc'].notna()].sort_values(by='event_date_utc', ascending=True)

In [ ]:
non_missing_trace_df = df_cleaned[df_cleaned['trace_id'].notna() & df_cleaned['event_date_utc'].notna()]
non_missing_trace_df.sort_values(by='event_date_utc', ascending=True)

The missingness is highly structured: for valid timestamps, events on Jul 27–30 have no trace ID, while Jul 31–Aug 2 have one, suggesting a source/schema evolution rather than random bad data. Here are some duplicated trace IDs

In [ ]:
dup_non_missing_trace_df = non_missing_trace_df[non_missing_trace_df.duplicated(subset=['trace_id'], keep=False)]
display(dup_non_missing_trace_df)

print(len(dup_non_missing_trace_df), "duplicated trace_id records found.")